# So sánh checkpoint QbH V3 và V5 trên gallery 300 bài

Notebook này đánh giá **hai checkpoint trên cùng một giao thức**: gallery gồm đúng 300 bản nhạc gốc; query lấy từ test split V3, chỉ giữ các query có audio và ánh xạ nhãn chắc chắn; nhãn và thứ hạng được tính ở cấp bài hát. Mỗi checkpoint dùng cấu hình tiền xử lý được lưu trong contract của checkpoint. Scorer cố định: local MaxMean + global max, fused = 0.5 local + 0.5 global.

## Chuẩn bị Kaggle

1. Bật GPU (T4/P100 đều được).
2. Gắn dataset chứa `dataset_split/` và `2_music_database/` (300 file MP3).
3. Gắn `results.zip` của V3 và `qbh_final_v5_train_outputs.zip` của V5 làm Kaggle inputs.
4. Chạy Run All. Nếu Kaggle đặt tên file khác, sửa ba biến đường dẫn ở cell cấu hình.

Notebook chỉ đánh giá, không train. Nó dùng các query test V3 có sẵn trong dữ liệu; query thiếu audio hoặc chưa ánh xạ chắc chắn tới tên bài trong gallery sẽ bị loại và ghi số lượng vào JSON. Bài chưa map vẫn được giữ làm distractor. Nếu test song bị dùng trong train V5, notebook sẽ dừng. Đây là closed-set retrieval, không phải đánh giá open-set hay DTW.


In [ ]:
import os, sys, json, csv, io, re, math, time, hashlib, zipfile, shutil, unicodedata, subprocess, gc
from pathlib import Path, PurePosixPath
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse
import soundfile as sf
import librosa
from tqdm.auto import tqdm
from IPython.display import display
try:
    import typeguard
    if not hasattr(typeguard, 'check_argument_types'):
        raise ImportError
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'typeguard==2.13.3'])
    for _name in list(sys.modules):
        if _name == 'typeguard' or _name.startswith('typeguard.'):
            del sys.modules[_name]


# Chỉ sửa các biến đường dẫn này nếu Kaggle không tự tìm thấy input.
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/qbh_v3_v5_300song')
V3_ZIP_OVERRIDE = None
V5_ZIP_OVERRIDE = None
DATASET_ROOT_OVERRIDE = None  # Thư mục cha có dataset_split/ và 2_music_database/.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Hãy bật GPU trong Kaggle Settings rồi chạy lại.')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)


In [ ]:
def unique_file(override, pattern, description):
    if override:
        p = Path(override).resolve()
        if not p.is_file(): raise FileNotFoundError(p)
        return p
    found = sorted({p.resolve() for p in INPUT_ROOT.rglob(pattern) if p.is_file()})
    if len(found) != 1:
        raise RuntimeError(f'{description}: cần đúng một file, tìm thấy {len(found)}; đặt override. {found[:8]}')
    return found[0]

def locate_member(zf, suffix):
    hits = [n for n in zf.namelist() if n.replace('\\', '/').endswith(suffix)]
    if len(hits) != 1: raise RuntimeError(f'ZIP cần đúng một member {suffix}; thấy {len(hits)}')
    return hits[0]

def extract_member(zpath, suffix, dest):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        member = locate_member(zf, suffix)
        with zf.open(member) as src, dest.open('wb') as out: shutil.copyfileobj(src, out)
    return dest

V3_ZIP = unique_file(V3_ZIP_OVERRIDE, 'results.zip', 'V3 results.zip')
V5_ZIP = unique_file(V5_ZIP_OVERRIDE, 'qbh_final_v5_train_outputs.zip', 'V5 train outputs ZIP')
DATASET_ROOTS = []
if DATASET_ROOT_OVERRIDE:
    DATASET_ROOTS = [Path(DATASET_ROOT_OVERRIDE).resolve()]
else:
    DATASET_ROOTS = sorted({p.parent.resolve() for p in INPUT_ROOT.rglob('dataset_split') if p.is_dir()})
    DATASET_ROOTS += [p.resolve() for p in INPUT_ROOT.rglob('*') if p.is_dir() and
                      (p / 'dataset_split').is_dir() and (p / '2_music_database').is_dir()]
    DATASET_ROOTS = list(dict.fromkeys(DATASET_ROOTS))
if len(DATASET_ROOTS) != 1:
    raise RuntimeError(f'Không xác định duy nhất dataset root; tìm thấy {len(DATASET_ROOTS)}. Đặt DATASET_ROOT_OVERRIDE.')
DATA_ROOT = DATASET_ROOTS[0]
SPLIT_ROOT = DATA_ROOT / 'dataset_split'
MUSIC_ROOT = DATA_ROOT / '2_music_database'
if not SPLIT_ROOT.is_dir() or not MUSIC_ROOT.is_dir(): raise FileNotFoundError('Dataset cần dataset_split/ và 2_music_database/.')
V3_DIR = OUTPUT_ROOT / 'inputs' / 'v3'; V5_DIR = OUTPUT_ROOT / 'inputs' / 'v5'
V3_CKPT = extract_member(V3_ZIP, 'checkpoints/best.pth', V3_DIR / 'best.pth')
V3_MANIFEST = extract_member(V3_ZIP, 'manifest.csv', V3_DIR / 'manifest.csv')
V3_CONTRACT = json.loads(extract_member(V3_ZIP, 'checkpoint_contract.json', V3_DIR / 'checkpoint_contract.json').read_text(encoding='utf-8'))
V5_CKPT = extract_member(V5_ZIP, 'qbh_final_v5_best.pth', V5_DIR / 'best.pth')
print('V3:', V3_ZIP.name, '| V5:', V5_ZIP.name, '| data:', DATA_ROOT)
print('Gallery MP3:', len(list(MUSIC_ROOT.glob('*.mp3'))))


In [ ]:
def norm_name(s):
    s = unicodedata.normalize('NFKD', str(s)).casefold().strip()
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = s.replace('&', 'and').replace('+', 'plus')
    return ''.join(c for c in s if c.isalnum())

mp3_by_name = defaultdict(list)
for p in MUSIC_ROOT.glob('*.mp3'): mp3_by_name[norm_name(p.stem)].append(p)
if len(list(MUSIC_ROOT.glob('*.mp3'))) != 300: raise RuntimeError('Gallery phải gồm đúng 300 file MP3.')
mapping_path = DATA_ROOT / 'song_name_mapping.json'
if not mapping_path.is_file():
    _mapping_candidates = list(INPUT_ROOT.rglob('song_name_mapping.json'))
    if len(_mapping_candidates) == 1: mapping_path = _mapping_candidates[0]
song_alias = {}
if mapping_path.is_file():
    title_id_path = mapping_path.with_name('song_title_to_id.json')
    if not title_id_path.is_file():
        raise FileNotFoundError(title_id_path)
    _mapping = json.loads(title_id_path.read_text(encoding='utf-8'))
    for _source_name, _song_id in _mapping.items():
        _key = norm_name(_source_name)
        if _key in song_alias and song_alias[_key] != _song_id: song_alias[_key] = None
        else: song_alias[_key] = _song_id

def map_song_name(name):
    _alias = song_alias.get(norm_name(name))
    _keys = [norm_name(_alias), norm_name(name)] if _alias else [norm_name(name)]
    _hits = {p for key in _keys for p in mp3_by_name.get(key, [])}
    return next(iter(_hits)) if len(_hits) == 1 else None

v3_manifest = pd.read_csv(V3_MANIFEST, dtype=str, keep_default_na=False)
paired_test = v3_manifest[(v3_manifest['split'] == 'test') & (v3_manifest['has_reference'].str.lower() == 'true')].copy()
if paired_test.empty: raise RuntimeError('V3 manifest không có paired test queries.')
gallery_paths = {p.stem: p for p in sorted(MUSIC_ROOT.glob('*.mp3'))}

# Index humming files by immutable audio_id; only rows from the locked V3 test split are used.
audio_index = {}
for p in SPLIT_ROOT.rglob('*.wav'):
    if p.stem in audio_index: raise RuntimeError(f'ID humming bị trùng: {p.stem}')
    audio_index[p.stem] = p
queries, missing_ids, unmapped_target_ids = [], [], set()
for row in paired_test.to_dict('records'):
    audio_id, sid = row['audio_id'], row['song_id']
    p = audio_index.get(audio_id)
    if p is None:
        missing_ids.append(audio_id); continue
    # Ensure the file belongs to the held-out test folder.
    if 'test' not in [part.casefold() for part in p.parts]:
        raise RuntimeError(f'Query {audio_id} không nằm trong thư mục test: {p}')
    target_path = map_song_name(row['folder_name'])
    if target_path is None:
        unmapped_target_ids.add(sid); continue
    queries.append({'key': audio_id, 'song': target_path.stem, 'song_id': sid, 'path': p})
if len(queries) < 700: raise RuntimeError(f'Chỉ còn {len(queries)} query; yêu cầu tối thiểu 700 để comparison có ý nghĩa.')
if len({q['song'] for q in queries}) < 40: raise RuntimeError(f'Phải còn ít nhất 40 test target songs; thấy {len({q["song"] for q in queries})}')

v3_state = torch.load(V3_CKPT, map_location='cpu', weights_only=True)
v5_state = torch.load(V5_CKPT, map_location='cpu', weights_only=True)
if not v3_state.get('model_state_dict') or not v5_state.get('model_state_dict'):
    raise RuntimeError('Checkpoint thiếu model_state_dict.')
v3_train = set((V3_CONTRACT.get('train_song_to_idx') or {}).keys())
v5_train = set(map(str, v5_state.get('train_songs', [])))
if not v3_train or not v5_train: raise RuntimeError('Checkpoint thiếu train song IDs; không thể kiểm tra test leakage.')
query_song_ids = {q['song_id'] for q in queries}
overlap_v3 = sorted(query_song_ids & v3_train)
overlap_v5 = sorted(query_song_ids & v5_train)
if overlap_v3 or overlap_v5:
    raise RuntimeError(f'Leakage: query test song xuất hiện trong train songs. V3={len(overlap_v3)}, V5={len(overlap_v5)}. Không dùng tập này để kết luận tổng quát hóa.')
print(f'Locked test: {len(paired_test)} paired rows; {len(queries)} query dùng được; thiếu audio={len(missing_ids)}; target chưa map={len(unmapped_target_ids)}; target songs={len({q["song"] for q in queries})}.')
print('Gallery: đúng 300 bài | test query song overlap train: 0')


In [ ]:
# CoverHunter backbone is embedded to keep this notebook self-contained.
import logging
import math
from typing import Union
from typeguard import check_argument_types

class BasicModel(torch.nn.Module):
  """Minimal CoverHunter base class required by the embedded inference model."""

  def model_size(self) -> int:
    return sum(parameter.numel() for parameter in self.parameters())



class Linear(torch.nn.Module):
  def __init__(self, in_dim, out_dim, bias=True, w_init_gain='linear'):
    super(Linear, self).__init__()
    self.linear_layer = torch.nn.Linear(in_dim, out_dim, bias=bias)

    torch.nn.init.xavier_uniform_(
      self.linear_layer.weight,
      gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, x):
    return self.linear_layer(x)


class Conv1d(torch.nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=1, stride=1,
               padding=None, dilation=1, bias=True, w_init_gain='linear'):
    super(Conv1d, self).__init__()
    if padding is None:
      assert (kernel_size % 2 == 1)
      padding = int(dilation * (kernel_size - 1) / 2)

    self.conv = torch.nn.Conv1d(in_channels, out_channels,
                                kernel_size=kernel_size, stride=stride,
                                padding=padding, dilation=dilation,
                                bias=bias)

    torch.nn.init.xavier_uniform_(
      self.conv.weight, gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, signal):
    conv_signal = self.conv(signal)
    return conv_signal


class CenterLoss(nn.Module):
  """Center loss.

  Reference:
    A Discriminative Feature Learning Approach for Deep Face Recognition.

  Args:
      num_classes (int): number of classes.
      feat_dim (int): feature dimension.
  """

  def __init__(self, num_classes=10, feat_dim=2, use_gpu=True):
    super(CenterLoss, self).__init__()
    self.num_classes = num_classes
    self.feat_dim = feat_dim
    self.use_gpu = use_gpu
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    if self.use_gpu:
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).cuda())
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).to("cuda"))
    else:
      self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim))
    logging.info("CenterLoss: num_classes({}), feat_dim({})".format(
      num_classes, feat_dim))
    return

  def forward(self, x, labels):
    """
    Args:
        x: feature matrix with shape (batch_size, feat_dim).
        labels: ground truth labels with shape (batch_size).
    """
    batch_size = x.size(0)
    distmat = torch.pow(x, 2).sum(dim=1, keepdim=True). \
                expand(batch_size, self.num_classes) + \
              torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(
                self.num_classes, batch_size).t()
    distmat.addmm_(1, -2, x, self.centers.t())

    classes = torch.arange(self.num_classes).long()
    if self.use_gpu:
      classes = classes.cuda()
    labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
    mask = labels.eq(classes.expand(batch_size, self.num_classes))

    dist = distmat * mask.float()
    loss = dist.clamp(min=1e-12, max=1e+12).sum() / batch_size

    return loss


class FocalLoss(nn.Module):
  """ Focal Loss implement for https://arxiv.org/abs/1708.02002 """

  def __init__(self, gamma: float = 2., alpha: Optional[List] = None, num_cls: int = -1,
               reduction: str = 'mean'):

    super(FocalLoss, self).__init__()
    assert check_argument_types()
    if reduction not in ['mean', 'sum']:
      raise NotImplementedError(
        'Reduction {} not implemented.'.format(reduction))
    self._reduction = reduction
    self._alpha = alpha
    self._gamma = gamma
    if alpha is not None:
      assert len(alpha) <= num_cls, "{} != {}".format(len(alpha), num_cls)
      self._alpha = torch.tensor(self._alpha)
    self._eps = np.finfo(float).eps
    logging.info("Init Focal loss with gamma:{}".format(gamma))
    return

  def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """ compute focal loss for pred and label

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss
    """
    b = y_pred.size(0)
    y_pred_softmax = torch.nn.Softmax(dim=1)(y_pred) + self._eps
    ce = -torch.log(y_pred_softmax)
    ce = ce.gather(1, y_true.view(-1, 1))

    y_pred_softmax = y_pred_softmax.gather(1, y_true.view(-1, 1))
    weight = torch.pow(torch.sub(1., y_pred_softmax), self._gamma)

    if self._alpha is not None:
      self._alpha = self._alpha.to(y_pred.device)
      alpha = self._alpha.gather(0, y_true.view(-1))
      alpha = alpha.unsqueeze(1)
      alpha = alpha / torch.sum(alpha) * b
      weight = torch.mul(alpha, weight)
    fl_loss = torch.mul(weight, ce).squeeze(1)
    return self._reduce(fl_loss)

  def forward_with_onehot(self, y_pred: torch.Tensor,
                          y_true: torch.Tensor) -> torch.Tensor:
    """ Another implement for "forward" with onehot label matrix.

    It is not good because when ce_embed size is large, more memory will used.

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss

    """
    y_pred = torch.nn.Softmax(dim=1)(y_pred)
    y_true = torch.nn.functional.one_hot(y_true, self.num_cls)

    eps = np.finfo(float).eps
    y_pred = y_pred + eps
    ce = torch.mul(y_true, -torch.log(y_pred))
    weight = torch.mul(y_true, torch.pow(torch.sub(1., y_pred), self._gamma))
    fl_loss = torch.mul(self.alpha, torch.mul(weight, ce))

    fl_loss, _ = torch.max(fl_loss, dim=1)
    return self._reduce(fl_loss)

  def _reduce(self, x):
    if self._reduction == 'mean':
      return torch.mean(x)
    else:
      return torch.sum(x)


class HardTripletLoss(nn.Module):
  """Hard/Hardest Triplet Loss
    (pytorch implementation of https://omoindrot.github.io/triplet-loss)
    For each anchor, we get the hardest positive and hardest negative to form a triplet.
    """

  def __init__(self, margin=0.1):
    """ Args:
      margin: margin for triplet loss
    """
    super(HardTripletLoss, self).__init__()
    self._margin = margin
    return

  def forward(self, embeddings, labels):
    """
    Args:
        labels: labels of the batch, of size (batch_size,)
        embeddings: tensor of shape (batch_size, embed_dim)

    Returns:
        triplet_loss: scalar tensor containing the triplet loss
    """
    pairwise_dist = self._pairwise_distance(embeddings, squared=False)

    mask_anchor_positive = self._get_anchor_positive_triplet_mask(
      labels).float()
    valid_positive_dist = pairwise_dist * mask_anchor_positive
    hardest_positive_dist, _ = torch.max(valid_positive_dist, dim=1,
                                         keepdim=True)

    # Get the hardest negative pairs
    mask_negative = self._get_anchor_negative_triplet_mask(labels).float()
    max_negative_dist, _ = torch.max(pairwise_dist, dim=1, keepdim=True)
    negative_dist = pairwise_dist + max_negative_dist * (1.0 - mask_negative)
    hardest_negative_dist, _ = torch.min(negative_dist, dim=1, keepdim=True)

    # Combine biggest d(a, p) and smallest d(a, n) into final triplet loss
    triplet_loss = F.relu(
      hardest_positive_dist - hardest_negative_dist + self._margin)
    triplet_loss = torch.mean(triplet_loss)
    return triplet_loss

  @staticmethod
  def _pairwise_distance(x, squared=False, eps=1e-16):
    # Compute the 2D matrix of distances between all the embeddings.

    cor_mat = torch.matmul(x, x.t())
    norm_mat = cor_mat.diag()
    distances = norm_mat.unsqueeze(1) - 2 * cor_mat + norm_mat.unsqueeze(0)
    distances = F.relu(distances)

    if not squared:
      mask = torch.eq(distances, 0.0).float()
      distances = distances + mask * eps
      distances = torch.sqrt(distances)
      distances = distances * (1.0 - mask)
    return distances

  @staticmethod
  def _get_anchor_positive_triplet_mask(labels):
    """Return a 2D mask where mask[a, p] is True, if a and p are distinct and
       have same label.

    """
    # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    device = labels.device
    indices_not_equal = torch.eye(labels.shape[0]).to(device).byte() ^ 1
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = indices_not_equal * labels_equal
    return mask

  @staticmethod
  def _get_anchor_negative_triplet_mask(labels):
    """Return a 2D mask where mask[a, n] is True iff a and n have distinct labels.
    """
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = labels_equal ^ 1
    return mask


def subsequent_chunk_mask(
    size: int,
    chunk_size: int,
    num_left_chunks: int = -1,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
  """Create mask for subsequent steps (size, size) with chunk size,
       this is for streaming encoder

    Args:
        size (int): size of mask
        chunk_size (int): size of chunk
        num_left_chunks (int): number of left chunks
            <0: use full chunk
            >=0: use num_left_chunks
        device (torch.device): "cpu" or "cuda" or torch.Tensor.device

    Returns:
        torch.Tensor: mask

    Examples:
        >>> subsequent_chunk_mask(4, 2)
        [[1, 1, 0, 0],
         [1, 1, 0, 0],
         [1, 1, 1, 1],
         [1, 1, 1, 1]]
    """
  ret = torch.zeros(size, size, device=device, dtype=torch.bool)
  for i in range(size):
    if num_left_chunks < 0:
      start = 0
    else:
      start = max((i // chunk_size - num_left_chunks) * chunk_size, 0)
    ending = min((i // chunk_size + 1) * chunk_size, size)
    ret[i, start:ending] = True
  return ret


def add_optional_chunk_mask(xs: torch.Tensor, masks: torch.Tensor,
                            use_dynamic_chunk: bool,
                            use_dynamic_left_chunk: bool,
                            decoding_chunk_size: int, static_chunk_size: int,
                            num_decoding_left_chunks: int):
  """ Apply optional mask for encoder.

    Args:
        masks:
        xs (torch.Tensor): padded input, (B, L, D), L for max length
        mask (torch.Tensor): mask for xs, (B, 1, L)
        use_dynamic_chunk (bool): whether to use dynamic chunk or not
        use_dynamic_left_chunk (bool): whether to use dynamic left chunk for
            training.
        decoding_chunk_size (int): decoding chunk size for dynamic chunk, it's
            0: default for training, use random dynamic chunk.
            <0: for decoding, use full chunk.
            >0: for decoding, use fixed chunk size as set.
        static_chunk_size (int): chunk size for static chunk training/decoding
            if it's greater than 0, if use_dynamic_chunk is true,
            this parameter will be ignored
        num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
            >=0: use num_decoding_left_chunks
            <0: use all left chunks

    Returns:
        torch.Tensor: chunk mask of the input xs.
    """
  # Whether to use chunk mask or not
  if use_dynamic_chunk:
    max_len = xs.size(1)
    if decoding_chunk_size < 0:
      chunk_size = max_len
      num_left_chunks = -1
    elif decoding_chunk_size > 0:
      chunk_size = decoding_chunk_size
      num_left_chunks = num_decoding_left_chunks
    else:
      # chunk size is either [1, 25] or full context(max_len).
      # Since we use 4 times subsampling and allow up to 1s(100 frames)
      # delay, the maximum frame is 100 / 4 = 25.
      chunk_size = torch.randint(1, max_len, (1,)).item()
      num_left_chunks = -1
      if chunk_size > max_len // 2:
        chunk_size = max_len
      else:
        chunk_size = chunk_size % 25 + 1
        if use_dynamic_left_chunk:
          max_left_chunks = (max_len - 1) // chunk_size
          num_left_chunks = torch.randint(0, max_left_chunks,
                                          (1,)).item()
    chunk_masks = subsequent_chunk_mask(xs.size(1), chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  elif static_chunk_size > 0:
    num_left_chunks = num_decoding_left_chunks
    chunk_masks = subsequent_chunk_mask(xs.size(1), static_chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  else:
    chunk_masks = masks
  return chunk_masks


def make_pad_mask(lengths: torch.Tensor, max_len: int = 0) -> torch.Tensor:
  """Make mask tensor containing indices of padded part.

    See description of make_non_pad_mask.

    Args:
        lengths (torch.Tensor): Batch of lengths (B,).
        max_len:

    Returns:
        torch.Tensor: Mask tensor containing indices of padded part.

    Examples:
        >>> make_pad_mask(torch.tensor([5, 3, 2]))
        masks = [[0, 0, 0, 0 ,0],
                 [0, 0, 0, 1, 1],
                 [0, 0, 1, 1, 1]]
    """
  batch_size = lengths.size(0)
  max_len = max_len if max_len > 0 else lengths.max().item()
  seq_range = torch.arange(0,
                           max_len,
                           dtype=torch.int64,
                           device=lengths.device)
  seq_range_expand = seq_range.unsqueeze(0).expand(batch_size, max_len)
  seq_length_expand = lengths.unsqueeze(-1)
  mask = seq_range_expand >= seq_length_expand
  return mask


class Swish(torch.nn.Module):
  """Construct an Swish object."""

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """Return Swish activation function."""
    return x * torch.sigmoid(x)


def get_activation(act):
  """Return activation function."""
  activation_funcs = {
    "hardtanh": torch.nn.Hardtanh,
    "tanh": torch.nn.Tanh,
    "relu": torch.nn.ReLU,
    "selu": torch.nn.SELU,
    "swish": getattr(torch.nn, "SiLU", Swish),
    "gelu": torch.nn.GELU
  }
  return activation_funcs[act]()


class PositionalEncoding(torch.nn.Module):
  """Positional encoding.

    :param int d_model: embedding dim
    :param float dropout_rate: dropout rate
    :param int max_len: maximum input length

    PE(pos, 2i)   = sin(pos/(10000^(2i/dmodel)))
    PE(pos, 2i+1) = cos(pos/(10000^(2i/dmodel)))
    """

  def __init__(self,
               d_model: int,
               dropout_rate: float,
               max_len: int = 5000,
               reverse: bool = False):
    """Construct an PositionalEncoding object."""
    super().__init__()
    self.d_model = d_model
    self.xscale = math.sqrt(self.d_model)
    self.dropout = torch.nn.Dropout(p=dropout_rate)
    self.max_len = max_len

    self.pe = torch.zeros(self.max_len, self.d_model)
    position = torch.arange(0, self.max_len,
                            dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
      torch.arange(0, self.d_model, 2, dtype=torch.float32) *
      -(math.log(10000.0) / self.d_model))
    self.pe[:, 0::2] = torch.sin(position * div_term)
    self.pe[:, 1::2] = torch.cos(position * div_term)
    self.pe = self.pe.unsqueeze(0)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Add positional encoding.

        Args:
            x (torch.Tensor): Input. Its shape is (batch, time, ...)
            offset (int, torch.tensor): position offset

        Returns:
            torch.Tensor: Encoded tensor. Its shape is (batch, time, ...)
            torch.Tensor: for compatibility to RelPositionalEncoding
        """

    self.pe = self.pe.to(x.device)
    pos_emb = self.position_encoding(offset, x.size(1), False)
    x = x * self.xscale + pos_emb
    return self.dropout(x), self.dropout(pos_emb)

  def position_encoding(self, offset: Union[int, torch.Tensor], size: int,
                        apply_dropout: bool = True) -> torch.Tensor:
    """ For getting encoding in a streaming fashion

        Attention!!!!!
        we apply dropout only once at the whole utterance level in a none
        streaming way, but will call this function several times with
        increasing input size in a streaming scenario, so the dropout will
        be applied several times.

        Args:
            apply_dropout:
            offset (int or torch.tensor): start offset
            size (int): required size of position encoding

        Returns:
            torch.Tensor: Corresponding encoding
        """
    # How to subscript a Union type:
    #   https://github.com/pytorch/pytorch/issues/69434
    if isinstance(offset, int):
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    elif isinstance(offset, torch.Tensor) and offset.dim() == 0:  # scalar
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    else:  # for batched streaming decoding on GPU
      assert torch.max(offset) + size < self.max_len
      index = offset.unsqueeze(1) + torch.arange(0, size).to(offset.device)  # B X T
      flag = index > 0
      # remove negative offset
      index = index * flag
      pos_emb = F.embedding(index, self.pe[0])  # B X T X d_model

    if apply_dropout:
      pos_emb = self.dropout(pos_emb)
    return pos_emb


class RelPositionalEncoding(PositionalEncoding):
  """Relative positional encoding module.
    See : Appendix B in https://arxiv.org/abs/1901.02860
    Args:
        d_model (int): Embedding dimension.
        dropout_rate (float): Dropout rate.
        max_len (int): Maximum input length.
    """

  def __init__(self, d_model: int, dropout_rate: float, max_len: int = 5000):
    """Initialize class."""
    super().__init__(d_model, dropout_rate, max_len, reverse=True)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute positional encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, `*`).
        Returns:
            torch.Tensor: Encoded tensor (batch, time, `*`).
            torch.Tensor: Positional embedding tensor (1, time, `*`).
        """
    self.pe = self.pe.to(x.device)
    x = x * self.xscale
    pos_emb = self.position_encoding(offset, x.size(1), False)
    return self.dropout(x), self.dropout(pos_emb)


class NoPositionalEncoding(torch.nn.Module):
  """ No position encoding
    """

  def __init__(self, d_model: int, dropout_rate: float):
    super().__init__()
    self.d_model = d_model
    self.dropout = torch.nn.Dropout(p=dropout_rate)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """ Just return zero vector for interface compatibility
        """
    pos_emb = torch.zeros(1, x.size(1), self.d_model).to(x.device)
    return self.dropout(x), pos_emb

  def position_encoding(
      self, offset: Union[int, torch.Tensor], size: int) -> torch.Tensor:
    return torch.zeros(1, size, self.d_model)


class BaseSubsampling(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.right_context = 0
    self.subsampling_rate = 1

  def position_encoding(self, offset: Union[int, torch.Tensor],
                        size: int) -> torch.Tensor:
    return self.pos_enc.position_encoding(offset, size)


class LinearNoSubsampling(BaseSubsampling):
  """Linear transform the input without subsampling

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an linear object."""
    super().__init__()
    self.out = torch.nn.Sequential(
      torch.nn.Linear(idim, odim),
      torch.nn.LayerNorm(odim, eps=1e-5),
      torch.nn.Dropout(dropout_rate),
    )
    self.pos_enc = pos_enc_class
    self.right_context = 0
    self.subsampling_rate = 1

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
    x = self.out(x)
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask


class Conv2dSubsampling4(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/4 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling4 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.out = torch.nn.Sequential(
      torch.nn.Linear(odim * (((idim - 1) // 2 - 1) // 2), odim))
    self.pos_enc = pos_enc_class
    # The right context for every conv layer is computed by:
    # (kernel_size - 1) * frame_rate_of_this_layer
    self.subsampling_rate = 4
    # 6 = (3 - 1) * 1 + (3 - 1) * 2
    self.right_context = 6

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 4.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 4.
            torch.Tensor: positional encoding

        """
    x = x.unsqueeze(1)  # (b, c=1, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.out(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2]


class Conv2dSubsampling8(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/8 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling8 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.linear = torch.nn.Linear(
      odim * ((((idim - 1) // 2 - 1) // 2 - 1) // 2), odim)
    self.pos_enc = pos_enc_class
    self.subsampling_rate = 8
    # 14 = (3 - 1) * 1 + (3 - 1) * 2 + (3 - 1) * 4
    self.right_context = 14

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 8.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 8.
            torch.Tensor: positional encoding
        """
    x = x.unsqueeze(1)  # (b, c, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.linear(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2][:, :, :-2:2]


class MultiHeadedAttention(nn.Module):
  """Multi-Head Attention layer.

    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, n_head: int, n_feat: int, dropout_rate: float):
    """Construct an MultiHeadedAttention object."""
    super().__init__()
    assert n_feat % n_head == 0
    # We assume d_v always equals d_k
    self.d_k = n_feat // n_head
    self.h = n_head
    self.linear_q = nn.Linear(n_feat, n_feat)
    self.linear_k = nn.Linear(n_feat, n_feat)
    self.linear_v = nn.Linear(n_feat, n_feat)
    self.linear_out = nn.Linear(n_feat, n_feat)
    self.dropout = nn.Dropout(p=dropout_rate)

  def forward_qkv(
      self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Transform query, key and value.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).

        Returns:
            torch.Tensor: Transformed query tensor, size
                (#batch, n_head, time1, d_k).
            torch.Tensor: Transformed key tensor, size
                (#batch, n_head, time2, d_k).
            torch.Tensor: Transformed value tensor, size
                (#batch, n_head, time2, d_k).

        """
    n_batch = query.size(0)
    q = self.linear_q(query).view(n_batch, -1, self.h, self.d_k)
    k = self.linear_k(key).view(n_batch, -1, self.h, self.d_k)
    v = self.linear_v(value).view(n_batch, -1, self.h, self.d_k)
    q = q.transpose(1, 2)  # (batch, head, time1, d_k)
    k = k.transpose(1, 2)  # (batch, head, time2, d_k)
    v = v.transpose(1, 2)  # (batch, head, time2, d_k)

    return q, k, v

  def forward_attention(
      self, value: torch.Tensor, scores: torch.Tensor,
      mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool)
  ) -> torch.Tensor:
    """Compute attention context vector.

        Args:
            value (torch.Tensor): Transformed value, size
                (#batch, n_head, time2, d_k).
            scores (torch.Tensor): Attention score, size
                (#batch, n_head, time1, time2).
            mask (torch.Tensor): Mask, size (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.

        Returns:
            torch.Tensor: Transformed value (#batch, time1, d_model)
                weighted by the attention score (#batch, time1, time2).

        """
    n_batch = value.size(0)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be True?
    #   1. onnx(16/4) [WHY? Because we feed real cache & real mask for the
    #           1st chunk to ease the onnx export.]
    #   2. pytorch training
    if mask.size(2) > 0:  # time2 > 0
      mask = mask.unsqueeze(1).eq(0)  # (batch, 1, *, time2)
      # For last chunk, time2 might be larger than scores.size(-1)
      mask = mask[:, :, :, :scores.size(-1)]  # (batch, 1, *, time2)
      scores = scores.masked_fill(mask, -float('inf'))
      attn = torch.softmax(scores, dim=-1).masked_fill(
        mask, 0.0)  # (batch, head, time1, time2)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be False?
    #   1. onnx(16/-1, -1/-1, 16/0)
    #   2. jit (16/-1, -1/-1, 16/0, 16/4)
    else:
      attn = torch.softmax(scores, dim=-1)  # (batch, head, time1, time2)

    p_attn = self.dropout(attn)
    x = torch.matmul(p_attn, value)  # (batch, head, time1, d_k)
    x = (x.transpose(1, 2).contiguous().view(n_batch, -1,
                                             self.h * self.d_k)
         )  # (batch, time1, d_model)

    return self.linear_out(x)  # (batch, time1, d_model)

  def forward(self, query: torch.Tensor, key: torch.Tensor,
              value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot product attention.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2).
                1.When applying cross attention between decoder and encoder,
                the batch padding mask for input is in (#batch, 1, T) shape.
                2.When applying self attention of encoder,
                the mask is in (#batch, T, T)  shape.
                3.When applying self attention of decoder,
                the mask is in (#batch, L, L)  shape.
                4.If the different position in decoder see different block
                of the encoder, such as Mocha, the passed in mask could be
                in (#batch, L, T) shape. But there is no such case in current
                Wenet.
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`


        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`

        """
    q, k, v = self.forward_qkv(query, key, value)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
    return self.forward_attention(v, scores, mask), new_cache


class RelPositionMultiHeadedAttention(MultiHeadedAttention):
  """Multi-Head Attention layer with relative position encoding.
    Paper: https://arxiv.org/abs/1901.02860
    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.
    """

  def __init__(self, n_head, n_feat, dropout_rate):
    """Construct an RelPositionMultiHeadedAttention object."""
    super().__init__(n_head, n_feat, dropout_rate)
    # linear transformation for positional encoding
    self.linear_pos = nn.Linear(n_feat, n_feat, bias=False)
    # these two learnable bias are used in matrix c and matrix d
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    self.pos_bias_u = nn.Parameter(torch.Tensor(self.h, self.d_k))
    self.pos_bias_v = nn.Parameter(torch.Tensor(self.h, self.d_k))
    torch.nn.init.xavier_uniform_(self.pos_bias_u)
    torch.nn.init.xavier_uniform_(self.pos_bias_v)

  def rel_shift(self, x, zero_triu: bool = False):
    """Compute relative positinal encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, size).
            zero_triu (bool): If true, return the lower triangular part of
                the matrix.
        Returns:
            torch.Tensor: Output tensor.
        """

    zero_pad = torch.zeros((x.size()[0], x.size()[1], x.size()[2], 1),
                           device=x.device,
                           dtype=x.dtype)
    x_padded = torch.cat([zero_pad, x], dim=-1)

    x_padded = x_padded.view(x.size()[0],
                             x.size()[1],
                             x.size(3) + 1, x.size(2))
    x = x_padded[:, :, 1:].view_as(x)

    if zero_triu:
      ones = torch.ones((x.size(2), x.size(3)))
      x = x * torch.tril(ones, x.size(3) - x.size(2))[None, None, :, :]

    return x

  def forward(self, query: torch.Tensor,
              key: torch.Tensor, value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute 'Scaled Dot Product Attention' with rel. positional encoding.
        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): Positional embedding tensor
                (#batch, time2, size).
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        """
    q, k, v = self.forward_qkv(query, key, value)
    q = q.transpose(1, 2)  # (batch, time1, head, d_k)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    n_batch_pos = pos_emb.size(0)
    p = self.linear_pos(pos_emb).view(n_batch_pos, -1, self.h, self.d_k)
    p = p.transpose(1, 2)  # (batch, head, time1, d_k)

    # (batch, head, time1, d_k)
    q_with_bias_u = (q + self.pos_bias_u).transpose(1, 2)
    # (batch, head, time1, d_k)
    q_with_bias_v = (q + self.pos_bias_v).transpose(1, 2)

    # compute attention score
    # first compute matrix a and matrix c
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    # (batch, head, time1, time2)
    matrix_ac = torch.matmul(q_with_bias_u, k.transpose(-2, -1))

    # compute matrix b and matrix d
    # (batch, head, time1, time2)
    matrix_bd = torch.matmul(q_with_bias_v, p.transpose(-2, -1))
    # Remove rel_shift since it is useless in speech recognition,
    # and it requires special attention for streaming.
    # matrix_bd = self.rel_shift(matrix_bd)

    scores = (matrix_ac + matrix_bd) / math.sqrt(
      self.d_k)  # (batch, head, time1, time2)

    return self.forward_attention(v, scores, mask), new_cache


class ConvolutionModule(nn.Module):
  """ConvolutionModule in Conformer model."""

  def __init__(self,
               channels: int,
               kernel_size: int = 15,
               activation: nn.Module = nn.ReLU(),
               norm: str = "batch_norm",
               causal: bool = False,
               bias: bool = True):
    """Construct an ConvolutionModule object.
        Args:
            channels (int): The number of channels of conv layers.
            kernel_size (int): Kernel size of conv layers.
            causal (int): Whether use causal convolution or not
        """
    assert check_argument_types()
    super().__init__()

    self.pointwise_conv1 = nn.Conv1d(
      channels,
      2 * channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    # self.lorder is used to distinguish if it's a causal convolution,
    # if self.lorder > 0: it's a causal convolution, the input will be
    #    padded with self.lorder frames on the left in forward.
    # else: it's a symmetrical convolution
    if causal:
      padding = 0
      self.lorder = kernel_size - 1
    else:
      # kernel_size should be an odd number for none causal convolution
      assert (kernel_size - 1) % 2 == 0
      padding = (kernel_size - 1) // 2
      self.lorder = 0
    self.depthwise_conv = nn.Conv1d(
      channels,
      channels,
      kernel_size,
      stride=1,
      padding=padding,
      groups=channels,
      bias=bias,
    )

    assert norm in ['batch_norm', 'layer_norm']
    if norm == "batch_norm":
      self.use_layer_norm = False
      self.norm = nn.BatchNorm1d(channels)
    else:
      self.use_layer_norm = True
      self.norm = nn.LayerNorm(channels)

    self.pointwise_conv2 = nn.Conv1d(
      channels,
      channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    self.activation = activation

  def forward(
      self,
      x: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      cache: torch.Tensor = torch.zeros((0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute convolution module.
        Args:
            x (torch.Tensor): Input tensor (#batch, time, channels).
            mask_pad (torch.Tensor): used for batch padding (#batch, 1, time),
                (0, 0, 0) means fake mask.
            cache (torch.Tensor): left context cache, it is only
                used in causal convolution (#batch, channels, cache_t),
                (0, 0, 0) meas fake cache.
        Returns:
            torch.Tensor: Output tensor (#batch, time, channels).
        """
    # exchange the temporal dimension and the feature dimension
    x = x.transpose(1, 2)  # (#batch, channels, time)

    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    if self.lorder > 0:
      if cache.size(2) == 0:  # cache_t == 0
        x = nn.functional.pad(x, (self.lorder, 0), 'constant', 0.0)
      else:
        assert cache.size(0) == x.size(0)  # equal batch
        assert cache.size(1) == x.size(1)  # equal channel
        x = torch.cat((cache, x), dim=2)
      assert (x.size(2) > self.lorder)
      new_cache = x[:, :, -self.lorder:]
    else:
      # It's better we just return None if no cache is required,
      # However, for JIT export, here we just fake one tensor instead of
      # None.
      new_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)

    # GLU mechanism
    x = self.pointwise_conv1(x)  # (batch, 2*channel, dim)
    x = nn.functional.glu(x, dim=1)  # (batch, channel, dim)

    # 1D Depthwise Conv
    x = self.depthwise_conv(x)
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.activation(self.norm(x))
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.pointwise_conv2(x)
    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    return x.transpose(1, 2), new_cache


class PositionwiseFeedForward(torch.nn.Module):
  """Positionwise feed forward layer.

    FeedForward are appied on each position of the sequence.
    The output dim is same with the input dim.

    Args:
        idim (int): Input dimenstion.
        hidden_units (int): The number of hidden units.
        dropout_rate (float): Dropout rate.
        activation (torch.nn.Module): Activation function
    """

  def __init__(self,
               idim: int,
               hidden_units: int,
               dropout_rate: float,
               activation: torch.nn.Module = torch.nn.ReLU()):
    """Construct a PositionwiseFeedForward object."""
    super(PositionwiseFeedForward, self).__init__()
    self.w_1 = torch.nn.Linear(idim, hidden_units)
    self.activation = activation
    self.dropout = torch.nn.Dropout(dropout_rate)
    self.w_2 = torch.nn.Linear(hidden_units, idim)

  def forward(self, xs: torch.Tensor) -> torch.Tensor:
    """Forward function.

        Args:
            xs: input tensor (B, L, D)
        Returns:
            output tensor, (B, L, D)
        """
    return self.w_2(self.dropout(self.activation(self.w_1(xs))))


class ConformerEncoderLayer(torch.nn.Module):
  """Encoder layer module.
    Args:
        size (int): Input dimension.
        self_attn (torch.nn.Module): Self-attention module instance.
            `MultiHeadedAttention` or `RelPositionMultiHeadedAttention`
            instance can be used as the argument.
        feed_forward (torch.nn.Module): Feed-forward module instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        feed_forward_macaron (torch.nn.Module): Additional feed-forward module
             instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        conv_module (torch.nn.Module): Convolution module instance.
            `ConvlutionModule` instance can be used as the argument.
        dropout_rate (float): Dropout rate.
        normalize_before (bool):
            True: use layer_norm before each sub-block.
            False: use layer_norm after each sub-block.
        concat_after (bool): Whether to concat attention layer's input and
            output.
            True: x -> x + linear(concat(x, att(x)))
            False: x -> x + att(x)
    """

  def __init__(
      self,
      size: int,
      self_attn: torch.nn.Module,
      feed_forward: Optional[nn.Module] = None,
      feed_forward_macaron: Optional[nn.Module] = None,
      conv_module: Optional[nn.Module] = None,
      dropout_rate: float = 0.1,
      normalize_before: bool = True,
      concat_after: bool = False,
  ):
    """Construct an EncoderLayer object."""
    super().__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.feed_forward_macaron = feed_forward_macaron
    self.conv_module = conv_module
    self.norm_ff = nn.LayerNorm(size, eps=1e-5)  # for the FNN module
    self.norm_mha = nn.LayerNorm(size, eps=1e-5)  # for the MHA module
    if feed_forward_macaron is not None:
      self.norm_ff_macaron = nn.LayerNorm(size, eps=1e-5)
      self.ff_scale = 0.5
    else:
      self.ff_scale = 1.0
    if self.conv_module is not None:
      self.norm_conv = nn.LayerNorm(size,
                                    eps=1e-5)  # for the CNN module
      self.norm_final = nn.LayerNorm(
        size, eps=1e-5)  # for the final output of the block
    self.dropout = nn.Dropout(dropout_rate)
    self.size = size
    self.normalize_before = normalize_before
    self.concat_after = concat_after
    if self.concat_after:
      self.concat_linear = nn.Linear(size + size, size)
    else:
      self.concat_linear = nn.Identity()

  def forward(
      self,
      x: torch.Tensor,
      mask: torch.Tensor,
      pos_emb: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
      cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute encoded features.

        Args:
            x (torch.Tensor): (#batch, time, size)
            mask (torch.Tensor): Mask tensor for the input (#batch, time，time),
                (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): positional encoding, must not be None
                for ConformerEncoderLayer.
            mask_pad (torch.Tensor): batch padding mask used for conv module.
                (#batch, 1，time), (0, 0, 0) means fake mask.
            att_cache (torch.Tensor): Cache tensor of the KEY & VALUE
                (#batch=1, head, cache_t1, d_k * 2), head * d_k == size.
            cnn_cache (torch.Tensor): Convolution cache in conformer layer
                (#batch=1, size, cache_t2)
        Returns:
            torch.Tensor: Output tensor (#batch, time, size).
            torch.Tensor: Mask tensor (#batch, time, time).
            torch.Tensor: att_cache tensor,
                (#batch=1, head, cache_t1 + time, d_k * 2).
            torch.Tensor: cnn_cahce tensor (#batch, size, cache_t2).
        """

    # whether to use macaron style
    if self.feed_forward_macaron is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_ff_macaron(x)
      x = residual + self.ff_scale * self.dropout(
        self.feed_forward_macaron(x))
      if not self.normalize_before:
        x = self.norm_ff_macaron(x)

    # multi-headed self-attention module
    residual = x
    if self.normalize_before:
      x = self.norm_mha(x)

    x_att, new_att_cache = self.self_attn(
      x, x, x, mask, pos_emb, att_cache)
    if self.concat_after:
      x_concat = torch.cat((x, x_att), dim=-1)
      x = residual + self.concat_linear(x_concat)
    else:
      x = residual + self.dropout(x_att)
    if not self.normalize_before:
      x = self.norm_mha(x)

    # convolution module
    # Fake new cnn cache here, and then change it in conv_module
    new_cnn_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)
    if self.conv_module is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_conv(x)
      x, new_cnn_cache = self.conv_module(x, mask_pad, cnn_cache)
      x = residual + self.dropout(x)

      if not self.normalize_before:
        x = self.norm_conv(x)

    # feed forward module
    residual = x
    if self.normalize_before:
      x = self.norm_ff(x)

    x = residual + self.ff_scale * self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x = self.norm_ff(x)

    if self.conv_module is not None:
      x = self.norm_final(x)

    return x, mask, new_att_cache, new_cnn_cache


class ConformerEncoder(torch.nn.Module):
  def __init__(
      self,
      input_size: int,
      output_size: int,
      attention_heads: int = 4,
      linear_units: int = 2048,
      num_blocks: int = 6,
      dropout_rate: float = 0.1,
      positional_dropout_rate: float = 0.1,
      attention_dropout_rate: float = 0.0,
      input_layer: str = "conv2d",
      pos_enc_layer_type: str = "rel_pos",
      normalize_before: bool = True,
      concat_after: bool = False,
      static_chunk_size: int = 0,
      use_dynamic_chunk: bool = False,
      global_cmvn: Optional[torch.nn.Module] = None,
      use_dynamic_left_chunk: bool = False,
      positionwise_conv_kernel_size: int = 1,
      macaron_style: bool = True,
      selfattention_layer_type: str = "rel_selfattn",
      activation_type: str = "swish",
      use_cnn_module: bool = True,
      cnn_module_kernel: int = 15,
      causal: bool = False,
      cnn_module_norm: str = "batch_norm",
  ):
    """
        Args:
            input_size (int): input dim
            output_size (int): dimension of attention
            attention_heads (int): the number of heads of multi head attention
            linear_units (int): the hidden units number of position-wise feed
                forward
            num_blocks (int): the number of decoder blocks
            dropout_rate (float): dropout rate
            attention_dropout_rate (float): dropout rate in attention
            positional_dropout_rate (float): dropout rate after adding
                positional encoding
            input_layer (str): input layer type.
                optional [linear, conv2d, conv2d6, conv2d8]
            pos_enc_layer_type (str): Encoder positional encoding layer type.
                opitonal [abs_pos, scaled_abs_pos, rel_pos, no_pos]
            normalize_before (bool):
                True: use layer_norm before each sub-block of a layer.
                False: use layer_norm after each sub-block of a layer.
            concat_after (bool): whether to concat attention layer's input
                and output.
                True: x -> x + linear(concat(x, att(x)))
                False: x -> x + att(x)
            static_chunk_size (int): chunk size for static chunk training and
                decoding
            use_dynamic_chunk (bool): whether use dynamic chunk size for
                training or not, You can only use fixed chunk(chunk_size > 0)
                or dyanmic chunk size(use_dynamic_chunk = True)
            global_cmvn (Optional[torch.nn.Module]): Optional GlobalCMVN module
            use_dynamic_left_chunk (bool): whether use dynamic left chunk in
                dynamic chunk training
        """
    assert check_argument_types()
    logging.info("Init Conformer layers::")
    super().__init__()
    self._output_size = output_size

    if pos_enc_layer_type == "abs_pos":
      pos_enc_class = PositionalEncoding
    elif pos_enc_layer_type == "rel_pos":
      pos_enc_class = RelPositionalEncoding
    elif pos_enc_layer_type == "no_pos":
      pos_enc_class = NoPositionalEncoding
    else:
      raise ValueError("unknown pos_enc_layer: " + pos_enc_layer_type)
    logging.info("Pos layers - {}".format(pos_enc_layer_type))

    if input_layer == "linear":
      subsampling_class = LinearNoSubsampling
    elif input_layer == "conv2d":
      subsampling_class = Conv2dSubsampling4
    elif input_layer == "conv2d8":
      subsampling_class = Conv2dSubsampling8
    else:
      raise ValueError("unknown input_layer: " + input_layer)
    logging.info("Input layers - {}".format(input_layer))

    self.global_cmvn = global_cmvn
    self.embed = subsampling_class(
      input_size,
      output_size,
      dropout_rate,
      pos_enc_class(output_size, positional_dropout_rate),
    )

    self.normalize_before = normalize_before
    self.after_norm = torch.nn.LayerNorm(output_size, eps=1e-5)
    self.static_chunk_size = static_chunk_size
    self.use_dynamic_chunk = use_dynamic_chunk
    self.use_dynamic_left_chunk = use_dynamic_left_chunk

    activation = get_activation(activation_type)

    # self-attention module definition
    if pos_enc_layer_type != "rel_pos":
      encoder_selfattn_layer = MultiHeadedAttention
    else:
      encoder_selfattn_layer = RelPositionMultiHeadedAttention
    encoder_selfattn_layer_args = (
      attention_heads,
      output_size,
      attention_dropout_rate,
    )
    logging.info("self attn args: {}".format(encoder_selfattn_layer_args))

    # feed-forward module definition
    positionwise_layer = PositionwiseFeedForward
    positionwise_layer_args = (
      output_size,
      linear_units,
      dropout_rate,
      activation,
    )
    logging.info("position-wise args: {}".format(positionwise_layer_args))

    # convolution module definition
    convolution_layer = ConvolutionModule
    convolution_layer_args = (output_size, cnn_module_kernel, activation,
                              cnn_module_norm, causal)

    self.encoders = torch.nn.ModuleList([
      ConformerEncoderLayer(
        output_size,
        encoder_selfattn_layer(*encoder_selfattn_layer_args),
        positionwise_layer(*positionwise_layer_args),
        positionwise_layer(
          *positionwise_layer_args) if macaron_style else None,
        convolution_layer(
          *convolution_layer_args) if use_cnn_module else None,
        dropout_rate,
        normalize_before,
        concat_after,
      ) for _ in range(num_blocks)
    ])
    return

  def output_size(self) -> int:
    return self._output_size

  def forward(
      self,
      xs: torch.Tensor,
      xs_lens: torch.Tensor,
      decoding_chunk_size: int = -1,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Embed positions in tensor.

        Args:
            xs: padded input tensor (B, T, D)
            xs_lens: input length (B)
            decoding_chunk_size: decoding chunk size for dynamic chunk
                0: default for training, use random dynamic chunk.
                <0: for decoding, use full chunk.
                >0: for decoding, use fixed chunk size as set.
            num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
                >=0: use num_decoding_left_chunks
                <0: use all left chunks
        Returns:
            encoder output tensor xs, and subsampled masks
            xs: padded output tensor (B, T' ~= T/subsample_rate, D)
            masks: torch.Tensor batch padding mask after subsample
                (B, 1, T' ~= T/subsample_rate)
        """
    T = xs.size(1)
    masks = ~make_pad_mask(xs_lens, T).unsqueeze(1)  # (B, 1, T)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    xs, pos_emb, masks = self.embed(xs, masks)
    mask_pad = masks  # (B, 1, T/subsample_rate)
    chunk_masks = add_optional_chunk_mask(xs, masks,
                                          self.use_dynamic_chunk,
                                          self.use_dynamic_left_chunk,
                                          decoding_chunk_size,
                                          self.static_chunk_size,
                                          num_decoding_left_chunks)
    for layer in self.encoders:
      xs, chunk_masks, _, _ = layer(xs, chunk_masks, pos_emb, mask_pad)
    if self.normalize_before:
      xs = self.after_norm(xs)
    # Here we assume the mask is not changed in encoder layers, so just
    # return the masks before encoder layers, and the masks will be used
    # for cross attention with decoder later
    return xs, masks

  def forward_chunk(
      self,
      xs: torch.Tensor,
      offset: int,
      required_cache_size: int,
      att_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      cnn_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      att_mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """ Forward just one chunk

        Args:
            xs (torch.Tensor): chunk input, with shape (b=1, time, mel-dim),
                where `time == (chunk_size - 1) * subsample_rate + \
                        subsample.right_context + 1`
            offset (int): current offset in encoder output time stamp
            required_cache_size (int): cache size required for next chunk
                compuation
                >=0: actual cache size
                <0: means all history cache is required
            att_cache (torch.Tensor): cache tensor for KEY & VALUE in
                transformer/conformer attention, with shape
                (elayers, head, cache_t1, d_k * 2), where
                `head * d_k == hidden-dim` and
                `cache_t1 == chunk_size * num_decoding_left_chunks`.
            cnn_cache (torch.Tensor): cache tensor for cnn_module in conformer,
                (elayers, b=1, hidden-dim, cache_t2), where
                `cache_t2 == cnn.lorder - 1`

        Returns:
            torch.Tensor: output of current input xs,
                with shape (b=1, chunk_size, hidden-dim).
            torch.Tensor: new attention cache required for next chunk, with
                dynamic shape (elayers, head, ?, d_k * 2)
                depending on required_cache_size.
            torch.Tensor: new conformer cnn cache required for next chunk, with
                same shape as the original cnn_cache.

        """
    assert xs.size(0) == 1
    # tmp_masks is just for interface compatibility
    tmp_masks = torch.ones(1,
                           xs.size(1),
                           device=xs.device,
                           dtype=torch.bool)
    tmp_masks = tmp_masks.unsqueeze(1)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    # NOTE(xcsong): Before embed, shape(xs) is (b=1, time, mel-dim)
    xs, pos_emb, _ = self.embed(xs, tmp_masks, offset)
    # NOTE(xcsong): After  embed, shape(xs) is (b=1, chunk_size, hidden-dim)
    elayers, cache_t1 = att_cache.size(0), att_cache.size(2)
    chunk_size = xs.size(1)
    attention_key_size = cache_t1 + chunk_size
    pos_emb = self.embed.position_encoding(
      offset=offset - cache_t1, size=attention_key_size)
    if required_cache_size < 0:
      next_cache_start = 0
    elif required_cache_size == 0:
      next_cache_start = attention_key_size
    else:
      next_cache_start = max(attention_key_size - required_cache_size, 0)
    r_att_cache = []
    r_cnn_cache = []
    for i, layer in enumerate(self.encoders):
      # NOTE(xcsong): Before layer.forward
      #   shape(att_cache[i:i + 1]) is (1, head, cache_t1, d_k * 2),
      #   shape(cnn_cache[i])       is (b=1, hidden-dim, cache_t2)
      xs, _, new_att_cache, new_cnn_cache = layer(
        xs, att_mask, pos_emb,
        att_cache=att_cache[i:i + 1] if elayers > 0 else att_cache,
        cnn_cache=cnn_cache[i] if cnn_cache.size(0) > 0 else cnn_cache
      )
      # NOTE(xcsong): After layer.forward
      #   shape(new_att_cache) is (1, head, attention_key_size, d_k * 2),
      #   shape(new_cnn_cache) is (b=1, hidden-dim, cache_t2)
      r_att_cache.append(new_att_cache[:, :, next_cache_start:, :])
      r_cnn_cache.append(new_cnn_cache.unsqueeze(0))
    if self.normalize_before:
      xs = self.after_norm(xs)

    # NOTE(xcsong): shape(r_att_cache) is (elayers, head, ?, d_k * 2),
    #   ? may be larger than cache_t1, it depends on required_cache_size
    r_att_cache = torch.cat(r_att_cache, dim=0)
    # NOTE(xcsong): shape(r_cnn_cache) is (e, b=1, hidden-dim, cache_t2)
    r_cnn_cache = torch.cat(r_cnn_cache, dim=0)

    return (xs, r_att_cache, r_cnn_cache)

  def forward_chunk_by_chunk(
      self,
      xs: torch.Tensor,
      decoding_chunk_size: int,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """ Forward input chunk by chunk with chunk_size like a streaming
            fashion

        Here we should pay special attention to computation cache in the
        streaming style forward chunk by chunk. Three things should be taken
        into account for computation in the current network:
            1. transformer/conformer encoder layers output cache
            2. convolution in conformer
            3. convolution in subsampling

        However, we don't implement subsampling cache for:
            1. We can control subsampling module to output the right result by
               overlapping input instead of cache left context, even though it
               wastes some computation, but subsampling only takes a very
               small fraction of computation in the whole model.
            2. Typically, there are several covolution layers with subsampling
               in subsampling module, it is tricky and complicated to do cache
               with different convolution layers with different subsampling
               rate.
            3. Currently, nn.Sequential is used to stack all the convolution
               layers in subsampling, we need to rewrite it to make it work
               with cache, which is not prefered.
        Args:
            xs (torch.Tensor): (1, max_len, dim)
            chunk_size (int): decoding chunk size
        """
    assert decoding_chunk_size > 0
    # The model is trained by static or dynamic chunk
    assert self.static_chunk_size > 0 or self.use_dynamic_chunk
    subsampling = self.embed.subsampling_rate
    context = self.embed.right_context + 1  # Add current frame
    stride = subsampling * decoding_chunk_size
    decoding_window = (decoding_chunk_size - 1) * subsampling + context
    num_frames = xs.size(1)
    att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    outputs = []
    offset = 0
    required_cache_size = decoding_chunk_size * num_decoding_left_chunks

    # Feed forward overlap input step by step
    for cur in range(0, num_frames - context + 1, stride):
      end = min(cur + decoding_window, num_frames)
      chunk_xs = xs[:, cur:end, :]
      (y, att_cache, cnn_cache) = self.forward_chunk(
        chunk_xs, offset, required_cache_size, att_cache, cnn_cache)
      outputs.append(y)
      offset += y.size(1)
    ys = torch.cat(outputs, 1)
    masks = torch.ones((1, 1, ys.size(1)), device=ys.device, dtype=torch.bool)
    return ys, masks


class AttentiveStatisticsPooling(torch.nn.Module):
  """This class implements an attentive statistic pooling layer for each channel.
  It returns the concatenated mean and std of the input tensor.

  Arguments
  ---------
  channels: int
      The number of input channels.
  output_channels: int
      The number of output channels.
  """

  def __init__(self, channels, output_channels):
    super().__init__()

    self._eps = 1e-12
    self._linear = Linear(channels * 3, channels)
    self._tanh = torch.nn.Tanh()
    self._conv = Conv1d(
      in_channels=channels, out_channels=channels, kernel_size=1
    )
    self._final_layer = torch.nn.Linear(channels * 2, output_channels,
                                        bias=False)
    logging.info("Init AttentiveStatisticsPooling with {}->{}".format(
      channels, output_channels))
    return

  @staticmethod
  def _compute_statistics(x: torch.Tensor,
                          m: torch.Tensor,
                          eps: float,
                          dim: int = 2):
    mean = (m * x).sum(dim)
    std = torch.sqrt((m * (x - mean.unsqueeze(dim)).pow(2)).sum(dim).clamp(eps))
    return mean, std

  def forward(self, x: torch.Tensor):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, L, C].
    """

    x = x.transpose(1, 2)
    L = x.shape[-1]
    lengths = torch.ones(x.shape[0], device=x.device)
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)
    total = mask.sum(dim=2, keepdim=True).float()

    mean, std = self._compute_statistics(x, mask / total, self._eps)
    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)
    attn = self._conv(self._tanh(self._linear(
      attn.transpose(1, 2)).transpose(1, 2)))

    attn = attn.masked_fill(mask == 0, float("-inf"))  # Filter out zero-padding
    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    pooled_stats = self._final_layer(torch.cat((mean, std), dim=1))
    return pooled_stats

  def forward_with_mask(self, x: torch.Tensor,
                        lengths: Optional[torch.Tensor] = None):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, C, L].
      lengths:
    """
    L = x.shape[-1]

    if lengths is None:
      lengths = torch.ones(x.shape[0], device=x.device)

    # Make binary mask of shape [N, 1, L]
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)

    # Expand the temporal context of the pooling layer by allowing the
    # self-attention to look at global properties of the utterance.

    # torch.std is unstable for backward computation
    # https://github.com/pytorch/pytorch/issues/4320
    total = mask.sum(dim=2, keepdim=True).float()
    mean, std = self._compute_statistics(x, mask / total, self._eps)

    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)

    # Apply layers
    attn = self.conv(self._tanh(self._linear(attn, lengths)))

    # Filter out zero-paddings
    attn = attn.masked_fill(mask == 0, float("-inf"))

    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    # Append mean and std of the batch
    pooled_stats = torch.cat((mean, std), dim=1)
    pooled_stats = pooled_stats.unsqueeze(2)
    return pooled_stats

  @staticmethod
  def length_to_mask(length: torch.Tensor,
                     max_len: Optional[int] = None,
                     dtype: Optional[torch.dtype] = None,
                     device: Optional[torch.device] = None):
    """Creates a binary mask for each sequence.

    Arguments
    ---------
    length : torch.LongTensor
        Containing the length of each sequence in the batch. Must be 1D.
    max_len : int
        Max length for the mask, also the size of the second dimension.
    dtype : torch.dtype, default: None
        The dtype of the generated mask.
    device: torch.device, default: None
        The device to put the mask variable.

    Returns
    -------
    mask : tensor
        The binary mask.

    Example
    -------
    """
    assert len(length.shape) == 1

    if max_len is None:
      max_len = length.max().long().item()  # using arange to generate mask
    mask = torch.arange(
      max_len, device=length.device, dtype=length.dtype
    ).expand(len(length), max_len) < length.unsqueeze(1)

    if dtype is None:
      dtype = length.dtype

    if device is None:
      device = length.device

    mask = torch.as_tensor(mask, dtype=dtype, device=device)
    return mask


class Model(BasicModel):
  """ Csi Backbone Model:
      Batch-norm
      Conformer-Encoder(x6 or x4)
      Global-Avg-Pool
      Resize and linear
      Bn-neck

  Loss:
    Focal-loss(ce) + Triplet-loss + Center-loss

  Args:
    hp: Dict with all parameters
  """

  def __init__(self, hp: Dict):
    super(BasicModel, self).__init__()
    self._hp = hp

    self._global_cmvn = torch.nn.BatchNorm1d(hp["input_dim"])
    self._encoder = ConformerEncoder(
      input_size=hp["input_dim"],
      output_size=hp["encoder"]["output_dims"],
      linear_units=hp["encoder"]["attention_dim"],
      num_blocks=hp["encoder"]["num_blocks"])

    if hp["encoder"]["output_dims"] != hp["embed_dim"]:
      self._embed_lo = torch.nn.Linear(hp["encoder"]["output_dims"],
                                       hp["embed_dim"])
    else:
      self._embed_lo = None

    # Bottleneck
    self._bottleneck = torch.nn.BatchNorm1d(hp["embed_dim"])
    self._bottleneck.bias.requires_grad_(False)  # no shift

    self._pool_layer = AttentiveStatisticsPooling(
      hp["embed_dim"], output_channels=hp["embed_dim"])
    self._ce_layer = torch.nn.Linear(
      hp["embed_dim"], hp["ce"]["output_dims"], bias=False)

    # Loss
    if "alpha" in hp["ce"].keys():
      alpha = np.load(hp["ce"]["alpha"])
      alpha = 1.0 / (alpha + 1)
      alpha = alpha / np.sum(alpha)
      logging.warning("use alpha with {}".format(len(alpha)))
    else:
      alpha = None
      logging.warning("Not use alpha")

    self._ce_loss = FocalLoss(alpha=alpha, gamma=self._hp["ce"]["gamma"],
                              num_cls=self._hp["ce"]["output_dims"])
    self._triplet_loss = HardTripletLoss(margin=hp["triplet"]["margin"])
    self._center_loss = CenterLoss(num_classes=self._hp["ce"]["output_dims"],
                                   feat_dim=hp["embed_dim"], use_gpu=torch.cuda.is_available())
    logging.info(
      "Model size: {:.3f}M \n".format(self.model_size() / 1000 / 1000))
    return

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """feat[b, frame_size, feat_size] -> embed[b, embed_dim]"""

    x = self._global_cmvn(x.transpose(1, 2)).transpose(1, 2)
    xs_lens = torch.full(
      [x.size(0)], fill_value=x.size(1), dtype=torch.long).to(x.device).long()
    x, _ = self._encoder(x, xs_lens=xs_lens, decoding_chunk_size=-1)
    x = self._pool_layer(x)
    return x

  @torch.jit.ignore
  def compute_loss(self, anchor: torch.Tensor, label: torch.Tensor) -> Tuple[
    torch.Tensor, Dict]:
    """compute ce and triplet loss"""
    f_t = self.forward(anchor)
    # print("f_t", f_t)

    f_i = self._bottleneck(f_t)
    ce_pred = self._ce_layer(f_i)
    # print("ce_pred", ce_pred)
    ce_loss = self._ce_loss(ce_pred, label)
    loss = ce_loss * self._hp["ce"]["weight"]
    loss_dict = {"ce_loss": ce_loss}

    tri_weight = self._hp["triplet"]["weight"]
    if tri_weight > 0.01:
      tri_loss = self._triplet_loss(f_t, label)
      loss = loss + tri_loss * tri_weight
      loss_dict.update({"tri_loss": tri_loss})

    cen_weight = self._hp["center"]["weight"]
    if cen_weight > 0.0:
      cen_loss = self._center_loss(f_t, label)
      loss = loss + cen_loss * cen_weight
      loss_dict.update({"cen_loss": cen_loss})
    return loss, loss_dict

  @torch.jit.ignore
  def inference(self, feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
      embed = self.forward(feat)
      embed_ce = self._ce_layer(embed)
    return embed, embed_ce

  @torch.jit.export
  def compute_embed(self, feat: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
      embed = self.forward(feat)
    return embed


In [ ]:
class AudioPreprocessor:
    """Tiền xử lý âm thanh: Resample về 16,000 Hz, chuẩn hóa biên độ và cắt Sliding Window 10s / hop 3s."""
    def __init__(self, sample_rate: int = 16000, chunk_duration: float = 10.0, hop_duration: float = 3.0):
        self.sample_rate = sample_rate
        self.chunk_duration = chunk_duration
        self.hop_duration = hop_duration
        self.chunk_samples = int(chunk_duration * sample_rate)
        self.hop_samples = int(hop_duration * sample_rate)

    def load_audio(self, audio_path: str) -> np.ndarray:
        """Nạp và chuẩn hóa âm thanh, không nuốt lỗi âm thầm."""
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Không tìm thấy file âm thanh: {audio_path}")
            
        if audio_path.lower().endswith('.silent'):
            return np.zeros(self.chunk_samples, dtype=np.float32)

        try:
            if audio_path.lower().endswith('.mp3'):
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            else:
                y, sr = sf.read(audio_path, always_2d=False, dtype='float32')
                if len(y.shape) > 1:
                    y = np.mean(y, axis=1)
                if sr != self.sample_rate:
                    y = librosa.resample(y, orig_sr=sr, target_sr=self.sample_rate)
        except Exception as e:
            # Fallback thu doc bang librosa neu soundfile gap loi dinh dang wav
            try:
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            except Exception as final_e:
                print(f"[!] Canh bao: Khong the doc file {audio_path}: {final_e}")
                raise RuntimeError(f"Không thể đọc file âm thanh: {audio_path}") from final_e

        y = np.asarray(y, dtype=np.float32).reshape(-1)
        if y.size == 0:
            raise ValueError(f"File audio rong: {audio_path}")
        if not np.isfinite(y).all():
            raise ValueError(f"Audio chua NaN/Inf: {audio_path}")
        max_val = float(np.max(np.abs(y)))
        if max_val > 1e-6:
            y = y / max_val
        return y

    def slice_into_chunks(self, y: np.ndarray) -> list:
        """Cắt âm thanh thành danh sách các đoạn 10 giây (hop 3s). Trả về ít nhất 1 đoạn."""
        if len(y) <= self.chunk_samples:
            return [np.pad(y, (0, self.chunk_samples - len(y)), mode='constant')]
        chunks = []
        start = 0
        for start in range(0, len(y) - self.chunk_samples + 1, self.hop_samples):
            chunks.append(y[start : start + self.chunk_samples])
        last_chunk_end = start + self.chunk_samples
        remaining_unseen = len(y) - last_chunk_end
        if remaining_unseen > (self.chunk_samples // 3):
            final_chunk = y[-self.chunk_samples:]
            if not np.array_equal(final_chunk, chunks[-1]):
                chunks.append(final_chunk)
        return chunks if chunks else [np.pad(y, (0, max(0, self.chunk_samples - len(y))), mode='constant')[:self.chunk_samples]]


In [ ]:
import scipy.sparse

def shorter(feat: np.ndarray, mean_size: int = 3) -> np.ndarray:
    """Nén trục thời gian của ma trận đặc trưng CQT theo bước mean_size (CoverHunter chuẩn)."""
    if mean_size <= 1:
        return feat
    cqt = feat.T
    height, length = cqt.shape
    new_len = int(length / mean_size)
    new_cqt = np.zeros((height, new_len), dtype=np.float32)
    for i in range(new_len):
        new_cqt[:, i] = cqt[:, i * mean_size : (i + 1) * mean_size].mean(axis=1)
    return new_cqt.T

class PyCqt:
    """CoverHunter CQT with the FFT and kernel projection executed on CUDA."""
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = None):
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA is required for GPU CQT. Enable Kaggle GPU T4 and restart the session.')
        self._hop_size = hop_size
        self.floor_db = -80.0
        self._sample_rate = sample_rate
        self._device = torch.device('cuda', torch.cuda.current_device())
        if not max_freq:
            max_freq = sample_rate // 2
        sparse_kernel = self._compute_cqt_kernel(sample_rate, octave_resolution, min_freq, max_freq)
        dense_kernel = sparse_kernel.toarray() if scipy.sparse.issparse(sparse_kernel) else np.asarray(sparse_kernel)
        # The 96 x 16,384 complex64 matrix is about 12.6 MiB on a T4.
        self._kernel = torch.as_tensor(dense_kernel.astype(np.complex64, copy=False), device=self._device)

    @staticmethod
    def _compute_cqt_kernel(sampling_frequency: int, octave_resolution: int, minimum_frequency: float, maximum_frequency: float):
        quality_factor = 1.0 / (pow(2.0, 1.0 / octave_resolution) - 1.0)
        number_frequencies = round(octave_resolution * np.log2(maximum_frequency / minimum_frequency))
        fft_length = int(pow(2, np.ceil(np.log2(quality_factor * sampling_frequency / minimum_frequency))))
        cqt_kernel = np.zeros((number_frequencies, fft_length), dtype=complex)
        for i in range(number_frequencies):
            frequency_value = minimum_frequency * pow(2.0, i / octave_resolution)
            window_length = 2 * round(quality_factor * sampling_frequency / frequency_value / 2.0) + 1
            temporal_kernel = (np.hamming(window_length)
                               * np.exp(2.0 * np.pi * 1j * quality_factor
                                        * np.arange(-(window_length - 1) / 2.0, (window_length - 1) / 2.0 + 1) / window_length)
                               / window_length)
            pad_width = int((fft_length - window_length + 1) / 2)
            cqt_kernel[i, pad_width : pad_width + window_length] = temporal_kernel
        cqt_kernel = np.fft.fft(cqt_kernel, axis=1)
        cqt_kernel[np.absolute(cqt_kernel) < 0.01] = 0
        cqt_kernel = scipy.sparse.csr_matrix(cqt_kernel)
        return np.conjugate(cqt_kernel) / fft_length

    @staticmethod
    @torch.inference_mode()
    def _compute_cqt_spec(audio_signal: np.ndarray, sampling_frequency: int, time_resolution: int, cqt_kernel, device):
        step_length = round(sampling_frequency / time_resolution)
        number_times = int(np.floor(len(audio_signal) / step_length))
        number_frequencies, fft_length = cqt_kernel.shape
        pad_left = int(np.ceil((fft_length - step_length) / 2.0))
        pad_right = int(np.floor((fft_length - step_length) / 2.0))
        if number_times <= 0:
            return torch.empty((number_frequencies, 0), dtype=torch.float32, device=device)

        # Decode/resample stays on CPU; framing, FFT, CQT projection and magnitude run on CUDA.
        audio_gpu = torch.as_tensor(np.asarray(audio_signal, dtype=np.float32), device=device)
        audio_gpu = F.pad(audio_gpu, (pad_left, pad_right))
        frames = audio_gpu.unfold(0, fft_length, step_length)
        if frames.shape[0] != number_times:
            raise RuntimeError(f'CUDA CQT frame count mismatch: expected {number_times}, got {frames.shape[0]}')

        cqt_gpu = torch.empty((number_frequencies, number_times), dtype=torch.float32, device=device)
        frame_batch = 128
        for start in range(0, number_times, frame_batch):
            end = min(start + frame_batch, number_times)
            spectra = torch.fft.fft(frames[start:end], dim=-1)
            response = torch.matmul(cqt_kernel, spectra.transpose(0, 1))
            cqt_gpu[:, start:end] = response.abs()
        return cqt_gpu

    def compute_cqt(self, signal_float: np.ndarray, feat_dim_first: bool = False) -> np.ndarray:
        time_resolution = int(round(1.0 / self._hop_size))
        cqt_gpu = self._compute_cqt_spec(signal_float, self._sample_rate, time_resolution, self._kernel, self._device)
        magnitude = cqt_gpu.clamp_min(1e-9)
        ref_value = magnitude.max().clamp_min(1e-4)
        cqt_gpu = 20.0 * torch.log10(magnitude) - 20.0 * torch.log10(ref_value)
        cqt_gpu = cqt_gpu.clamp(min=self.floor_db, max=0.0) if self.floor_db is not None else cqt_gpu.clamp(max=0.0)
        if not feat_dim_first:
            cqt_gpu = cqt_gpu.transpose(0, 1)
        # Keep cached features on CPU to avoid pinning the whole dataset in VRAM.
        return cqt_gpu.contiguous().cpu().numpy()

class CoverHunterCQT:
    """
    Trích xuất đặc trưng CQT chuẩn CoverHunter:
    - 96 bins (12 bins/octave, 8 octaves từ 32 Hz đến 8000 Hz)
    - hop_size: 0.04s (25 fps), nén thời gian shorter(mean_size=3)
    - Chuẩn hóa dB theo biên độ cực đại: 20 * log10(cqt / max_cqt)
    """
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = 8000.0, mean_size: int = 3):
        self.sample_rate = sample_rate
        self.mean_size = mean_size
        self.py_cqt = PyCqt(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=octave_resolution,
            min_freq=min_freq,
            max_freq=max_freq
        )

    def extract(self, audio: np.ndarray) -> torch.Tensor:
        if isinstance(audio, torch.Tensor):
            audio = audio.detach().cpu().numpy()
        feat = self.py_cqt.compute_cqt(audio, feat_dim_first=False) # (T, 96)
        if self.mean_size > 1:
            feat = shorter(feat, mean_size=self.mean_size)
        return torch.from_numpy(feat).float()
class CQTExtractor:
    """Trích xuất Constant-Q Transform (CQT) hỗ trợ cả chế độ Fixed-Length (83 frames) và Variable-Length."""
    def __init__(self, sample_rate: int = 16000, n_bins: int = 96, bins_per_octave: int = 12, hop_size: float = 0.04, mean_size: int = 3, target_frames: int = 83):
        self.sample_rate = sample_rate
        self.n_bins = n_bins
        self.bins_per_octave = bins_per_octave
        self.hop_size = hop_size
        self.mean_size = mean_size
        self.target_frames = target_frames
        self.cqt_impl = CoverHunterCQT(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=bins_per_octave,
            min_freq=32.0,
            max_freq=8000.0,
            mean_size=mean_size
        )

    def extract(self, y: np.ndarray, variable_length: bool = False) -> torch.Tensor:
        """
        Trích xuất tensor CQT từ tín hiệu âm thanh 1D.
        - variable_length=False: Căn chỉnh về đúng [1, 96, target_frames] (mặc định 83 frames ~ 10s).
        - variable_length=True: Giữ nguyên độ dài thời gian thực tế [1, 96, T] phục vụ Stage 2 DTW.
        """
        feat = self.cqt_impl.extract(y)  # (T, 96)
        tensor = feat.T.unsqueeze(0)      # (1, 96, T)

        if not variable_length and self.target_frames is not None and self.target_frames > 0:
            curr_t = tensor.shape[-1]
            min_val = tensor.min()
            if curr_t < self.target_frames:
                tensor = F.pad(tensor, (0, self.target_frames - curr_t), mode='constant', value=min_val.item())
            elif curr_t > self.target_frames:
                tensor = tensor[:, :, :self.target_frames]
        return tensor



In [ ]:
def apply_cqt_pitch_shift(cqt: torch.Tensor, shift_bins: int) -> torch.Tensor:
    """
    Dịch chuyển cao độ trên ma trận CQT bằng cách tịnh tiến trục tần số (dim -2).
    - shift_bins > 0: Dịch lên cao độ cao hơn (tăng semitone).
    - shift_bins < 0: Dịch xuống cao độ thấp hơn (giảm semitone).
    - Điền các bin trống bằng cqt.min() (mức sàn im lặng), KHÔNG điền 0.0 (vốn là 0 dB cực đại).
    """
    if shift_bins == 0:
        return cqt

    min_val = cqt.min()
    shifted = torch.full_like(cqt, fill_value=min_val)
    num_bins = cqt.shape[-2]

    if shift_bins > 0:
        if shift_bins < num_bins:
            shifted[..., shift_bins:, :] = cqt[..., :num_bins - shift_bins, :]
    else:
        abs_shift = abs(shift_bins)
        if abs_shift < num_bins:
            shifted[..., :num_bins - abs_shift, :] = cqt[..., abs_shift:, :]

    return shifted
class CQTPitchShiftAugmentation:
    """
    Tăng cường dữ liệu dịch tone ngẫu nhiên (Full-Octave Pitch-Shift Data Augmentation).
    Dịch ngẫu nhiên trong dải [-max_shift_bins, +max_shift_bins] nửa cung (semitones).
    Mặc định dải ±12 semitones bao trọn 1 quãng 8 (chuyển đổi hoàn hảo giữa giọng Nam và Nữ).
    """
    def __init__(self, max_shift_bins: int = 12, prob: float = 0.8):
        self.max_shift_bins = max_shift_bins
        self.prob = prob

    def __call__(self, cqt_tensor: torch.Tensor) -> torch.Tensor:
        if random.random() > self.prob:
            return cqt_tensor
        shift = random.randint(-self.max_shift_bins, self.max_shift_bins)
        return apply_cqt_pitch_shift(cqt_tensor, shift)
class SpectrogramTempoAndMaskDistortion:
    """Nonlinear time warp plus SpecAugment masks; the CQT frequency axis is never warped."""
    def __init__(self, time_scale_range=(0.5, 1.5), max_mask_t=0, max_mask_f=0, prob=0.6, knots=6):
        self.scale_range = time_scale_range
        self.max_mask_t = int(max_mask_t)
        self.max_mask_f = int(max_mask_f)
        self.prob = prob
        self.knots = knots
        if not 0 < time_scale_range[0] <= time_scale_range[1] or knots < 2:
            raise ValueError('Invalid distortion parameters')
        if self.max_mask_t < 0 or self.max_mask_f < 0:
            raise ValueError('Mask widths must be non-negative')

    def _axis(self, size, batch, device, dtype):
        increments = torch.empty(batch, 1, self.knots, device=device, dtype=dtype).uniform_(*self.scale_range)
        positions = torch.cat([torch.zeros(batch, 1, 1, device=device, dtype=dtype), increments.cumsum(-1)], -1)
        positions = 2 * positions / positions[..., -1:] - 1
        return F.interpolate(positions, size=size, mode='linear', align_corners=True)[:, 0]

    @staticmethod
    def _mask_axis(x, axis, max_width):
        if max_width <= 0 or x.shape[axis] <= 1:
            return
        width = random.randint(1, min(max_width, x.shape[axis] - 1))
        start = random.randint(0, x.shape[axis] - width)
        slices = [slice(None)] * x.ndim
        slices[axis] = slice(start, start + width)
        fill_value = x.amin(dim=(-2, -1), keepdim=True)
        x[tuple(slices)] = fill_value

    def __call__(self, spec):
        if random.random() >= self.prob:
            return spec
        was_unbatched = spec.ndim == 3
        x = spec.unsqueeze(0) if was_unbatched else spec
        b, _, h, w = x.shape
        tx = self._axis(w, b, x.device, x.dtype)
        fy = torch.linspace(-1, 1, h, device=x.device, dtype=x.dtype).expand(b, h)
        grid = torch.stack([tx[:, None, :].expand(b, h, w), fy[:, :, None].expand(b, h, w)], -1)
        out = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True).clone()
        self._mask_axis(out, axis=-1, max_width=self.max_mask_t)
        self._mask_axis(out, axis=-2, max_width=self.max_mask_f)
        return out.squeeze(0) if was_unbatched else out


In [ ]:
from torch.autograd import Function

class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_val: float):
        ctx.lambda_val = lambda_val
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    """Lớp đảo ngược gradient với hệ số lambda điều chỉnh động theo epoch."""
    def __init__(self, lambda_val: float = 0.2):
        super(GradientReversalLayer, self).__init__()
        self.lambda_val = lambda_val

    def set_lambda(self, lambda_val: float):
        self.lambda_val = lambda_val

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return GradientReversalFunction.apply(x, self.lambda_val)
class LocalDomainClassifier(nn.Module):
    """
    Bộ phân loại miền cục bộ (Local Domain Classifier) theo kiến trúc ByteHum:
    - Nhận đầu vào: Chuỗi đặc trưng cục bộ x [B, T, 128] từ Conformer encoder.
    - Áp dụng các tầng tích chập 1D theo trục thời gian để học đặc trưng miền cục bộ.
    - Dự đoán phân phối nhãn miền (0: Vocal phong thu, 1: Tiếng humming) trên từng phân đoạn thời gian.
    """
    def __init__(self, feature_dim: int = 128, hidden_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(feature_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, 128] -> transpose thành [B, 128, T] cho Conv1D
        x_t = x.transpose(1, 2)
        feat = self.conv(x_t) # [B, hidden_dim, T]
        # Frame-level logits: [B, T]
        frame_logits = self.fc(feat.transpose(1, 2)).squeeze(-1)
        # Tổng hợp logits qua trục thời gian T
        return frame_logits.mean(dim=-1) # [B]
def build_coverhunter_model(ckpt_path: str = None, device: str = "cpu", allow_random_init: bool = False):
    """
    Khởi tạo mô hình CoverHunter Conformer chính thức (9.91M tham số, 6 khối Conformer, Attention Pooling 128-D).
    Nạp trọn vẹn 265/265 tensor từ checkpoint SHS100K pre-trained.
    """
    # Cấu hình siêu tham số mặc định chuẩn xác của CoverHunter SHS100K
    hp = {
        'input_dim': 96, 'embed_dim': 128,
        'encoder': {'output_dims': 128, 'num_blocks': 6, 'attention_dim': 256},
        'pool_type': 'attention',
        'ce': {'output_dims': 30000, 'weight': 1.0, 'gamma': 2},
        'triplet': {'margin': 0.3, 'weight': 0.1},
        'center': {'weight': 0.0}
    }

    ch_model = Model(hp)

    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state_dict = ckpt.get("generator", ckpt.get("model_state_dict", ckpt))
        ch_model.load_state_dict(state_dict)
        total_p = sum(p.numel() for p in ch_model.parameters())
        print(f"[+] Nạp thành công 265/265 tensors ({total_p:,} params) từ CoverHunter Pretrained:\n    -> {ckpt_path}")
    elif allow_random_init:
        print("[*] Khoi tao backbone tam thoi; checkpoint fine-tuned se duoc nap ngay sau do.")
    else:
        raise FileNotFoundError(f"Bat buoc phai co checkpoint Stage 1 SHS100K: {ckpt_path}")

    ch_model = ch_model.to(device)
    return ch_model
class LocalAlignmentClassifier(nn.Module):
    """
    Local Alignment Classifier (LAC) theo kiến trúc ByteHum (ICASSP 2024):
    - Trọng số Prototype: W in R^{K x L x C}, trong đó K là số lớp bài hát (train songs),
      L là số phân đoạn prototype cục bộ (mặc định L=4), C=128.
    - Điểm tương đồng MaxMean: S(X, W_k) = (1/L) * sum_{l=1}^L max_{t=1}^T cos(x_t, w_{k,l}).
    - Cho phép căn chỉnh cục bộ bất biến thời gian và trôi dạt nhịp điệu giữa Humming và Vocal.
    """
    def __init__(self, num_classes: int = 210, feature_dim: int = 128, num_prototypes: int = 4):
        super().__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_prototypes = num_prototypes
        # Khởi tạo ma trận prototype W [K, L, C]
        self.prototypes = nn.Parameter(torch.randn(num_classes, num_prototypes, feature_dim) * 0.05)
        self.temperature = nn.Parameter(torch.tensor(10.0))

    def forward(self, local_feat: torch.Tensor) -> torch.Tensor:
        # local_feat: [B, T, 128]
        # Chuẩn hóa L2 trên trục đặc trưng 128-D
        x_norm = F.normalize(local_feat, p=2, dim=-1) # [B, T, C]
        w_norm = F.normalize(self.prototypes, p=2, dim=-1) # [K, L, C]

        # Tính cosine similarity giữa toàn bộ T frames và K*L prototypes qua phép nhân ma trận:
        w_flat = w_norm.view(-1, self.feature_dim).t() # [C, K*L]
        sim = torch.matmul(x_norm, w_flat) # [B, T, K*L]
        B, T, _ = sim.shape
        sim = sim.view(B, T, self.num_classes, self.num_prototypes) # [B, T, K, L]

        # Phep toan MaxMean: Max theo truc thoi gian T, Mean theo so prototypes L
        max_t, _ = sim.max(dim=1) # [B, K, L]
        logits = max_t.mean(dim=-1) # [B, K]
        return logits * self.temperature.clamp(min=1.0, max=100.0)

class ConformerByteHumModel(nn.Module):
    """
    MÔ HÌNH HOÀN CHỈNH CONFORMER-BYTEHUM CHO QBH DOMAIN ADAPTATION:
    - Backbone: Conformer (6 khối, Conv2d Subsampling 4x -> T_enc=20 frames cho 10s audio)
    - Output 1: global_emb [B, 128] (qua Attentive Statistics Pooling + L2 Normalize)
    - Output 2: local_feat [B, T=20, 128] (chuỗi đặc trưng cục bộ trước pooling)
    - Output 3: lac_logits [B, K] (Local Alignment Classifier phân loại bài hát qua MaxMean)
    - Output 4: domain_logits [B] (Local Domain Classifier phân biệt miền Vocal vs Humming)
    """
    def __init__(self, ckpt_path: str = None, input_dim: int = 96, embedding_dim: int = 128, num_classes: int = 210, lambda_grl: float = 0.2, allow_random_init: bool = False):
        super().__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        if embedding_dim not in (128, 256, 512):
            raise ValueError('embedding_dim must be 128, 256 or 512')
        self.local_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)
        self.global_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)

        # Nạp backbone Conformer pre-trained từ CoverHunter Stage 1
        self.feature_extractor = build_coverhunter_model(ckpt_path=ckpt_path, allow_random_init=allow_random_init)
        
        # Đóng băng các tầng phụ không dùng của Stage 1 (ce_layer, center_loss)
        if hasattr(self.feature_extractor, '_ce_layer'):
            for p in self.feature_extractor._ce_layer.parameters(): p.requires_grad = False
        if hasattr(self.feature_extractor, '_center_loss'):
            for p in self.feature_extractor._center_loss.parameters(): p.requires_grad = False

        self.grl = GradientReversalLayer(lambda_val=lambda_grl)
        self.domain_classifier = LocalDomainClassifier(feature_dim=embedding_dim, hidden_dim=64)
        self.lac = LocalAlignmentClassifier(num_classes=num_classes, feature_dim=embedding_dim, num_prototypes=4)

    def train(self, mode: bool = True):
        super().train(mode)
        # Preserve the SHS100K CMVN statistics while fine-tuning on humming audio.
        if mode and hasattr(self.feature_extractor, '_global_cmvn'):
            self.feature_extractor._global_cmvn.eval()
        return self

    def set_lambda(self, lambda_val: float):
        self.grl.set_lambda(lambda_val)

    def _prepare_input(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4: x = x.squeeze(1)
        if x.dim() == 3:
            if x.shape[1] == 96 and x.shape[2] != 96: x = x.transpose(1, 2)
        elif x.dim() == 2:
            if x.shape[0] == 96 and x.shape[1] != 96: x = x.t().unsqueeze(0)
            else: x = x.unsqueeze(0)
        return x

    def extract_backbone_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trích xuất chuỗi đặc trưng cục bộ [B, T=20, 128] từ Conformer encoder trước ASP pooling."""
        x_prep = self._prepare_input(x)
        x_cmvn = self.feature_extractor._global_cmvn(x_prep.transpose(1, 2)).transpose(1, 2)
        xs_lens = torch.full([x_cmvn.size(0)], fill_value=x_cmvn.size(1), dtype=torch.long, device=x_cmvn.device)
        x_enc, _ = self.feature_extractor._encoder(x_cmvn, xs_lens=xs_lens, decoding_chunk_size=-1)
        return x_enc # [B, T=20, 128]

    def extract_local_features(self, x):
        return self.local_projection(self.extract_backbone_features(x))

    def extract_global_embedding(self, local_feat: torch.Tensor) -> torch.Tensor:
        """Chiếu chuỗi đặc trưng cục bộ qua Attentive Statistics Pooling thành Global 128-D Embedding."""
        emb = self.feature_extractor._pool_layer(local_feat)
        return F.normalize(self.global_projection(emb), p=2, dim=-1)

    def extract_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Hàm tiện ích trích xuất trực tiếp Global Embedding 128-D từ phổ CQT đầu vào."""
        return self.extract_global_embedding(self.extract_backbone_features(x))

    def forward(self, x: torch.Tensor, compute_lac: bool = True, compute_domain: bool = True):
        """Return global/local features and only the auxiliary heads requested by the run."""
        backbone_feat = self.extract_backbone_features(x)
        local_feat = self.local_projection(backbone_feat)
        global_emb = self.extract_global_embedding(backbone_feat)
        lac_logits = self.lac(local_feat) if compute_lac else None
        domain_logits = self.domain_classifier(self.grl(local_feat)) if compute_domain else None
        return global_emb, local_feat, lac_logits, domain_logits

    def freeze_lower_layers(self, num_blocks_to_freeze: int = 3, freeze_frontend: bool = True):
        """Đóng băng tầng frontend và các khối Conformer đầu tiên theo chiến lược Deep Freeze."""
        if freeze_frontend:
            if hasattr(self.feature_extractor, '_global_cmvn'):
                for p in self.feature_extractor._global_cmvn.parameters(): p.requires_grad = False
            if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'embed'):
                for p in self.feature_extractor._encoder.embed.parameters(): p.requires_grad = False

        if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'encoders'):
            for i, block in enumerate(self.feature_extractor._encoder.encoders):
                if i < num_blocks_to_freeze:
                    for p in block.parameters(): p.requires_grad = False
                else:
                    for p in block.parameters(): p.requires_grad = True

        if hasattr(self.feature_extractor, '_pool_layer'):
            for p in self.feature_extractor._pool_layer.parameters(): p.requires_grad = True
        for p in self.domain_classifier.parameters(): p.requires_grad = True
        for p in self.lac.parameters(): p.requires_grad = True

In [ ]:
# Các hàm inference/retrieval dùng chung scorer cho cả hai checkpoint.
audio_pre = AudioPreprocessor(sample_rate=16000, chunk_duration=10.0, hop_duration=3.0)
cqt = CQTExtractor(sample_rate=16000, n_bins=96, bins_per_octave=12,
                   hop_size=0.04, mean_size=3, target_frames=83)

def model_from_checkpoint(state, name):
    weights = state['model_state_dict']
    proto = weights.get('lac.prototypes')
    if proto is None or proto.ndim != 3:
        raise RuntimeError(f'{name}: không tìm thấy LAC prototypes [songs,4,128].')
    model = ConformerByteHumModel(ckpt_path=None, num_classes=int(proto.shape[0]),
                                  embedding_dim=128, allow_random_init=True)
    missing, unexpected = model.load_state_dict(weights, strict=False)
    if missing or unexpected:
        raise RuntimeError(f'{name}: checkpoint/model không tương thích; missing={missing[:8]}, unexpected={unexpected[:8]}')
    model = model.to(DEVICE).eval()
    return model

def file_cqts(path, floor_db):
    cqt.cqt_impl.py_cqt.floor_db = floor_db
    y = audio_pre.load_audio(str(path))
    chunks = audio_pre.slice_into_chunks(y)
    out = []
    for chunk in chunks:
        x = cqt.extract(chunk)
        if floor_db is not None: x = torch.clamp(x, min=float(floor_db))
        out.append(x.squeeze(0))
    return torch.stack(out)

@torch.inference_mode()
def encode_file(model, path, floor_db, batch_size=32):
    spec = file_cqts(path, floor_db)
    gs, ls = [], []
    for start in range(0, len(spec), batch_size):
        x = spec[start:start+batch_size].unsqueeze(1).to(DEVICE)
        with torch.autocast('cuda', dtype=torch.float16):
            g, l, _, _ = model(x, compute_lac=False, compute_domain=False)
        gs.append(F.normalize(g.float(), dim=-1).cpu())
        ls.append(F.normalize(l.float(), dim=-1).reshape(-1, 128).cpu())
    g, l = torch.cat(gs), torch.cat(ls)
    if not torch.isfinite(g).all() or not torch.isfinite(l).all(): raise RuntimeError(f'Non-finite features: {path}')
    return g, l

@torch.inference_mode()
def encode_gallery(model, floor_db, tag):
    result = {}
    for sid, path in tqdm(gallery_paths.items(), desc=f'{tag}: encode 300-song gallery'):
        result[sid] = encode_file(model, path, floor_db)
    return result

@torch.inference_mode()
def scores_for_query(model, gallery, query, floor_db):
    qg, ql = encode_file(model, query['path'], floor_db)
    qg, ql = qg.to(DEVICE), ql.to(DEVICE)
    songs = list(gallery)
    local_values, global_values = [], []
    # Batched song blocks avoid hundreds of tiny CUDA launches per query.
    for start in range(0, len(songs), 16):
        group = songs[start:start+16]
        gseq = [gallery[s][0] for s in group]
        lseq = [gallery[s][1] for s in group]
        gp = nn.utils.rnn.pad_sequence(gseq, batch_first=True).to(DEVICE)
        gmask = torch.arange(gp.shape[1], device=DEVICE)[None, :] < torch.tensor([len(x) for x in gseq], device=DEVICE)[:, None]
        gsim = torch.einsum('qd,scd->sqc', qg, gp).masked_fill(~gmask[:, None, :], float('-inf'))
        global_values.append(gsim.amax(dim=(1,2)).cpu())
        lp = nn.utils.rnn.pad_sequence(lseq, batch_first=True).to(DEVICE)
        lmask = torch.arange(lp.shape[1], device=DEVICE)[None, :] < torch.tensor([len(x) for x in lseq], device=DEVICE)[:, None]
        lsim = torch.einsum('qd,sld->sql', ql, lp).masked_fill(~lmask[:, None, :], float('-inf'))
        local_values.append(lsim.amax(dim=-1).mean(dim=-1).cpu())
    local = dict(zip(songs, torch.cat(local_values).tolist()))
    global_ = dict(zip(songs, torch.cat(global_values).tolist()))
    fused = {sid: 0.5 * (local[sid] + global_[sid]) for sid in gallery}
    return {'local': local, 'global': global_, 'fused': fused}

def expected_rank_metrics(score_map, target):
    vals = np.asarray(list(score_map.values()), dtype=np.float64)
    ids = list(score_map)
    t = vals[ids.index(target)]
    tol = np.maximum(1e-7, 1e-6 * np.maximum(np.abs(vals), abs(t)))
    ahead = int(np.sum(vals > t + tol))
    tied = max(1, int(np.sum(np.abs(vals - t) <= tol)))
    rr = sum(1.0 / r for r in range(ahead + 1, ahead + tied + 1)) / tied
    return {'rr': float(rr), **{f'hit@{k}': float(min(max(k-ahead, 0), tied) / tied) for k in (1,5,10,20)}}

def evaluate_model(state, name, floor_db):
    model = model_from_checkpoint(state, name)
    gallery = encode_gallery(model, floor_db, name)
    rows = []
    for q in tqdm(queries, desc=f'{name}: score queries'):
        score_sets = scores_for_query(model, gallery, q, floor_db)
        row = {'audio_id': q['key'], 'song_id': q['song']}
        for mode, smap in score_sets.items():
            row.update({f'{mode}_{k}': v for k, v in expected_rank_metrics(smap, q['song']).items()})
        rows.append(row)
    del model, gallery
    gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows)

# V3 contract has no dB floor; V5 run contract specifies -80 dB.
v5_contract = v5_state.get('run_contract', {})
if not v5_state.get('finished', False): raise RuntimeError('V5 BEST checkpoint is not finalized.')
if v5_state.get('checkpoint_schema') != 2: raise RuntimeError('V5 checkpoint schema is not the expected schema 2.')
v5_floor = v5_contract.get('preprocessing', {}).get('floor_db', -80.0)
v3_floor = V3_CONTRACT.get('preprocessing', {}).get('floor_db')
print('Preprocessing floor dB | V3:', v3_floor, '| V5:', v5_floor)


In [ ]:
started = time.perf_counter()
v3_rows = evaluate_model(v3_state, 'V3', v3_floor)
v5_rows = evaluate_model(v5_state, 'V5', v5_floor)
if v3_rows['audio_id'].tolist() != v5_rows['audio_id'].tolist(): raise RuntimeError('Query lists differ between models.')

def summarize(frame):
    out = {}
    for mode in ('local', 'global', 'fused'):
        out[mode] = {
            'query_mrr': float(frame[f'{mode}_rr'].mean()),
            'macro_song_mrr': float(frame.groupby('song_id')[f'{mode}_rr'].mean().mean()),
            **{f'hit@{k}': float(frame[f'{mode}_hit@{k}'].mean()) for k in (1,5,10,20)},
        }
    return out

v3_metrics, v5_metrics = summarize(v3_rows), summarize(v5_rows)
comparison = v3_rows.merge(v5_rows, on=['audio_id','song_id'], suffixes=('_v3','_v5'), validate='one_to_one')
for metric in ['fused_rr','fused_hit@1','fused_hit@5','fused_hit@10','fused_hit@20']:
    comparison[f'delta_{metric}'] = comparison[f'{metric}_v5'] - comparison[f'{metric}_v3']

# Song-cluster bootstrap CI for the paired macro-song MRR delta.
song_scores = comparison.groupby('song_id')[['fused_rr_v3', 'fused_rr_v5']].mean()
song_delta = (song_scores['fused_rr_v5'] - song_scores['fused_rr_v3']).to_numpy()
rng = np.random.default_rng(42)
boot = np.asarray([rng.choice(song_delta, size=len(song_delta), replace=True).mean() for _ in range(2000)])
report = {
  'protocol': 'paired V3 test queries x exact 300 original-song gallery; local MaxMean + global max; fused 0.5/0.5',
  'gallery_songs': len(gallery_paths), 'manifest_paired_test_queries': len(paired_test),
  'evaluated_common_queries': len(queries), 'missing_query_files': len(missing_ids),
  'test_songs': len({q['song'] for q in queries}), 'unmapped_target_songs': len(unmapped_target_ids),
  'v3_checkpoint': str(V3_ZIP.name), 'v5_checkpoint_archive': str(V5_ZIP.name),
  'v3_preprocessing': V3_CONTRACT.get('preprocessing'), 'v5_preprocessing': v5_contract.get('preprocessing'),
  'metrics': {'V3': v3_metrics, 'V5': v5_metrics,
              'delta_v5_minus_v3': {m: v5_metrics['fused'][m]-v3_metrics['fused'][m] for m in v3_metrics['fused']}},
  'paired_macro_song_mrr_delta_v5_minus_v3_ci95': np.quantile(boot, [0.025,0.975]).tolist(),
  'overlap_test_with_v3_train_songs': len(overlap_v3), 'overlap_test_with_v5_train_songs': len(overlap_v5),
  'elapsed_seconds': time.perf_counter()-started,
  'caveat': 'Different checkpoint training runs; result compares complete checkpoints on this common test protocol, not a controlled one-module ablation.'
}
v3_rows.to_csv(OUTPUT_ROOT / 'v3_query_metrics.csv', index=False)
v5_rows.to_csv(OUTPUT_ROOT / 'v5_query_metrics.csv', index=False)
comparison.to_csv(OUTPUT_ROOT / 'paired_query_comparison.csv', index=False)
(OUTPUT_ROOT / 'comparison_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame({'V3 fused': v3_metrics['fused'], 'V5 fused': v5_metrics['fused'],
                      'Delta': report['metrics']['delta_v5_minus_v3']}))
print('Paired macro-song MRR delta 95% CI:', report['paired_macro_song_mrr_delta_v5_minus_v3_ci95'])
print('Saved artifacts to:', OUTPUT_ROOT)
